Take each zarr file produced by racmo_2km_downscaled/download_upload_to_scratch.ipynb, rechunk to 1 in he time dimension and append to one large zarr for each 

In [1]:
import os
import s3fs
import fsspec
import boto3
import xarray as xr
import geopandas as gpd
import xarray as xr
import pandas as pd

In [18]:
client.shutdown()

2026-05-28 17:08:08,904 - distributed.nanny - WARNING - Worker process still alive after 4.0 seconds, killing
2026-05-28 17:08:08,906 - distributed.nanny - WARNING - Worker process still alive after 4.0 seconds, killing
2026-05-28 17:08:08,937 - distributed.nanny - WARNING - Worker process still alive after 4.0 seconds, killing
2026-05-28 17:08:08,956 - distributed.nanny - WARNING - Worker process still alive after 4.0 seconds, killing
2026-05-28 17:08:08,968 - distributed.nanny - WARNING - Worker process still alive after 4.0 seconds, killing


In [3]:
from dask.distributed import Client
client = Client()
client.cluster.scale(5)
client

Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: /user/jkingslake/load%20NCs/proxy/8787/status,
Dashboard: /user/jkingslake/load%20NCs/proxy/8787/status,Workers: 4
Total threads: 4,Total memory: 14.54 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:42811,Workers: 4
Dashboard: /user/jkingslake/load%20NCs/proxy/8787/status,Total threads: 4
Started: Just now,Total memory: 14.54 GiB
Comm: tcp://127.0.0.1:46039,Total threads: 1
Dashboard: /user/jkingslake/load%20NCs/proxy/38647/status,Memory: 3.63 GiB
Nanny: tcp://127.0.0.1:42795,


In [4]:
df = pd.read_csv('record_of_zarrs.csv')

In [5]:
df_subset = df[(df["dataset"] == "ff10m")]
df_subset

,Unnamed: 0,dataset,zarr_name,zarr_path,size_mb,n_files,has_zmetadata
1,1,ff10m,ff10m.1979_JFM.BN_RACMO2.3p2_ANT27_ERA5_3h.AIS...,s3://nasa-cryo-scratch/jkingslake/Daily-2km-za...,146.697116,17,True
4,4,ff10m,ff10m.1979_AMJ.BN_RACMO2.3p2_ANT27_ERA5_3h.AIS...,s3://nasa-cryo-scratch/jkingslake/Daily-2km-za...,894.754988,33,True
9,9,ff10m,ff10m.1979_JAS.BN_RACMO2.3p2_ANT27_ERA5_3h.AIS...,s3://nasa-cryo-scratch/jkingslake/Daily-2km-za...,902.776493,33,True
13,13,ff10m,ff10m.1979_OND.BN_RACMO2.3p2_ANT27_ERA5_3h.AIS...,s3://nasa-cryo-scratch/jkingslake/Daily-2km-za...,901.520007,33,True
17,17,ff10m,ff10m.1980_JFM.BN_RACMO2.3p2_ANT27_ERA5_3h.AIS...,s3://nasa-cryo-scratch/jkingslake/Daily-2km-za...,894.843136,33,True
...,...,...,...,...,...,...,...
735,735,ff10m,ff10m.2024_OND.BN_RACMO2.3p2_ANT27_ERA5_3h.AIS...,s3://nasa-cryo-scratch/jkingslake/Daily-2km-za...,902.559971,33,True
739,739,ff10m,ff10m.2025_JFM.BN_RACMO2.3p2_ANT27_ERA5_3h.AIS...,s3://nasa-cryo-scratch/jkingslake/Daily-2km-za...,883.954481,32,True
742,742,ff10m,ff10m.2025_AMJ.BN_RACMO2.3p2_ANT27_ERA5_3h.AIS...,s3://nasa-cryo-scratch/jkingslake/Daily-2km-za...,892.898760,33,True
746,746,ff10m,ff10m.2025_JAS.BN_RACMO2.3p2_ANT27_ERA5_3h.AIS...,s3://nasa-cryo-scratch/jkingslake/Daily-2km-za...,900.778713,33,True


In [6]:
import xarray as xr
from tqdm import tqdm

VAR_NAME = "ff10m"  # change this
DATASET  = "ff10m"      # change this

df_subset = df[df["dataset"] == DATASET]
f1_sorted = df_subset['zarr_path'].to_list()

OUT_PATH = f"s3://nasa-cryo-scratch/jkingslake/Daily-2km-zarr/{VAR_NAME}_all_06.zarr"

# Write first file
print(f"Writing first file: {f1_sorted[0]}")
ds = xr.open_zarr(f1_sorted[0], consolidated=True, chunks={})
ds = ds.chunk(time=1)
for var in ds.data_vars:
    if 'chunks' in ds[var].encoding:
        del ds[var].encoding['chunks']
ds.to_zarr(OUT_PATH, mode='w', zarr_format=2)
ds.close()

# Append remaining files
for path in tqdm(f1_sorted[1:], desc="Appending files"):
    ds = xr.open_zarr(path, consolidated=True, chunks={})
    ds = ds.chunk(time=1)
    for var in ds.data_vars:
        if 'chunks' in ds[var].encoding:
            del ds[var].encoding['chunks']
    ds.to_zarr(OUT_PATH, mode='a', append_dim='time', zarr_format=2)
    ds.close()

print(f"Done! Written to {OUT_PATH}")

Writing first file: s3://nasa-cryo-scratch/jkingslake/Daily-2km-zarr/ff10m/ff10m.1979_JFM.BN_RACMO2.3p2_ANT27_ERA5_3h.AIS.2km.DD.zarr


Appending files: 100%|██████████| 187/187 [1:29:34<00:00, 28.74s/it]

Done! Written to s3://nasa-cryo-scratch/jkingslake/Daily-2km-zarr/ff10m_all_06.zarr
